# ComfyUI Story Generator — API + Loop

**CONFIG hücresinde her şey yüklenir** (token + workflow JSON + referans görsel), sonra arka tarafta otomatik akar.

Sıra:
1. **CONFIG** — token + workflow JSON + referans görsel upload
2. **PROMPTS** — sabit prompt'lar + ACTIONS + seed
3. **Ortam kurulumu (ComfyUI + Manager)**
4. **IPAdapter custom node**
5. **IPAdapter + CLIP Vision indir (HF)**
6. **Checkpoint indir (Civitai, curl)**
7. **Dosya boyutu kontrolü**
8. **ComfyUI'yi arka planda başlat** (ref görseli input/'a kopyalar)
9. **Loop'la üret**
10. **Sonuçları göster**
11. **Zip'le indir**

## 1) CONFIG — token + workflow + referans görsel

Çalıştırınca sırayla iki dosya seçici açılır:
- Önce **workflow JSON**
- Sonra **referans görsel**

Yenisini yüklemek için: `!rm /content/workflow.json /content/ref/*`

In [ ]:
# @title 🔑 Tokens & uploads
import os, shutil
from google.colab import files
from IPython.display import Image, display

CIVITAI_TOKEN = "c2f8440d15d825229098e9a5a75c5dfa"  #@param {type:"string"}
HF_TOKEN      = ""  #@param {type:"string"}
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}

os.environ['CIVITAI_TOKEN'] = CIVITAI_TOKEN
os.environ['HF_TOKEN']      = HF_TOKEN

WORKSPACE = '/content/drive/MyDrive/ComfyUI' if USE_GOOGLE_DRIVE else '/content/ComfyUI'
os.environ['WORKSPACE'] = WORKSPACE

# === Workflow JSON upload ===
WORKFLOW_PATH = '/content/workflow.json'
if os.path.exists(WORKFLOW_PATH):
    print(f"✓ Workflow zaten mevcut: {WORKFLOW_PATH}")
else:
    print("📂 (1/2) Workflow JSON dosyanı seç:")
    uploaded = files.upload()
    if uploaded:
        src = list(uploaded.keys())[0]
        shutil.move(src, WORKFLOW_PATH)
        print(f"✓ Workflow yüklendi: {WORKFLOW_PATH}")
os.environ['WORKFLOW_PATH'] = WORKFLOW_PATH

# === Referans görsel upload ===
# Geçici olarak /content/ref/ klasörüne koyuyoruz, ComfyUI klasörüne DOKUNMUYORUZ.
# 8. hücre (ComfyUI başlat) bunu ComfyUI/input/'a kopyalayacak.
REF_STAGING_DIR = '/content/ref'
os.makedirs(REF_STAGING_DIR, exist_ok=True)

existing_refs = [f for f in os.listdir(REF_STAGING_DIR) if not f.startswith('.')]
if existing_refs:
    ref_name = existing_refs[0]
    ref_path = f'{REF_STAGING_DIR}/{ref_name}'
    print(f"✓ Referans görsel zaten yüklü: {ref_path}")
    os.environ['REF_FILENAME'] = ref_name
    os.environ['REF_STAGING_PATH'] = ref_path
    display(Image(ref_path, width=300))
else:
    print("\n📂 (2/2) Referans görselini seç:")
    uploaded = files.upload()
    if uploaded:
        src = list(uploaded.keys())[0]
        dest = f'{REF_STAGING_DIR}/{src}'
        shutil.move(src, dest)
        os.environ['REF_FILENAME'] = src
        os.environ['REF_STAGING_PATH'] = dest
        print(f"✓ Referans yüklendi: {dest}")
        print(f"   Workflow'a yazılacak dosya adı: {src}")
        display(Image(dest, width=300))

print(f"\nWORKSPACE     = {WORKSPACE}")
print(f"WORKFLOW_PATH = {WORKFLOW_PATH}")
print(f"REF_FILENAME  = {os.environ.get('REF_FILENAME', '(yok)')}")
print(f"CIVITAI_TOKEN = {'(set, ' + str(len(CIVITAI_TOKEN)) + ' chars)' if CIVITAI_TOKEN else '(BOŞ — checkpoint inmez!)'}")

## 2) PROMPTS — workflow beslemeleri

In [ ]:
QUALITY = "masterpiece, best quality, very aesthetic, absurdres, newest, highres, ultra detailed, extremely detailed, sharp focus, intricate details, perfect anatomy, perfect proportions, perfect hands, detailed face, detailed eyes, beautiful lighting, cinematic lighting, professional artwork, 8k, high resolution, Adult"

CHARACTER = "solo, 1girl, teal hair, long hair, twintails, green eyes, fair skin, slim, large breasts, smile"

BACKGROUND = "bedroom, white bedsheets, wrinkled bedsheets, large window, pink curtains, wooden furniture, wooden dresser, potted plant, indoor plant, sunny day, golden hour, warm sunlight, sunlight rays, soft shadows, cozy atmosphere, depth of field, blurry background, bokeh"

NEGATIVE = "worst quality, low quality, normal quality, lowres, jpeg artifacts, bad anatomy, bad hands, bad proportions, extra fingers, missing fingers, deformed, ugly, blurry, distorted, malformed, watermark, signature, text, amateur, sketch, unfinished"

ACTIONS = [
    "from above, close up, portrait, look at viewer, hetero, penis, doggystyle, uncensored, pov, wide open mouth,head grab,wet open mouth and Sticking out long wet tongue,sucking penis,",
    "feet, ass, pussy, from above, from behind, close up, portrait, hetero, penis, doggystyle, uncensored, pov",
    "feet, ass, pussy, from above, from behind, close up, portrait, hetero, penis, doggystyle, uncensored, pov, smug smile,",
    "topless, nipples, from above, close up, portrait, look at viewer, hetero, penis, doggystyle, uncensored, pov, wide open mouth,pov, headgrab,wet open mouth and Sticking out long wet tongue,sucking penis,",
    "vaginal, portrait, look at viewer, hetero, penis, cowgirl, uncensored, pov, smug smile,",
    "ass, pussy, from above, from behind, close up, portrait, hetero, penis, doggystyle, uncensored, pov,",
    "ass, pussy, from above, from behind, close up, portrait, hetero, penis, doggystyle, uncensored, pov, blush, annoyed, open mouth, cum, vaginal,, ass grab,",
    "feet, ass, pussy, from above, from behind, close up, portrait, hetero, penis, doggystyle, uncensored, pov, Thigh grab,",
    "feet, ass, pussy, from above, from behind, close up, portrait, hetero, penis, doggystyle, uncensored, pov, cum, waist grab,",
    "from above, close up, portrait, look at viewer, hetero, penis, { kneeling | allfours | sitting | squatting }, uncensored, pov, wide open mouth,pov, headgrab,wet open mouth and Sticking out wet tongue, deepthroat, irrumatio,",
    "pov, headgrab, sucking penis, caressing testicles, handjob, oral sex, deepthroat, irrumatio, veined penis, deep oral insertion,",
    "full nelson, 1boy, penis, sex, ass, blush, scared, open mouth, ejaculation,",
    "full nelson, 1boy, penis, sex, ass, blush, confused,",
]

SEED = 42

print(f"✓ {len(ACTIONS)} aksiyon → {len(ACTIONS)} görsel")
print(f"✓ Seed: {SEED} (sabit)")

## 3) Ortam kurulumu (ComfyUI + Manager)

Boş veya yarım klasör varsa (`.git` yoksa) otomatik silinip baştan klone eder.

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')

if WORKSPACE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive
else:
    %cd /content

# .git yoksa boş/bozuk klasör var demek → sil ve baştan klone et
![ ! -d $WORKSPACE/.git ] && echo -= Initial setup ComfyUI =- && rm -rf $WORKSPACE && git clone https://github.com/comfyanonymous/ComfyUI $WORKSPACE
%cd $WORKSPACE

!echo -= Updating ComfyUI =-
!git pull

!echo -= Install ComfyUI requirements =-
!pip install -r requirements.txt
!pip install blake3

# ComfyUI-Manager
%cd custom_nodes
![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
%cd ComfyUI-Manager
!git pull

%cd $WORKSPACE
!pip install GitPython
!python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

## 4) IPAdapter custom node

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
%cd $WORKSPACE/custom_nodes

![ ! -d ComfyUI_IPAdapter_plus ] && git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus
%cd ComfyUI_IPAdapter_plus
!git pull
![ -f requirements.txt ] && pip install -r requirements.txt

%cd $WORKSPACE

## 5) IPAdapter + CLIP Vision indir (HuggingFace)
Sorunsuz iniyor, login gerekmiyor.

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
%cd $WORKSPACE

!mkdir -p models/ipadapter models/clip_vision models/checkpoints/sdxl

!wget -c https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors \
  -O models/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors

!wget -c https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors \
  -O models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

!echo "--- IPAdapter ---" && ls -lh models/ipadapter/
!echo "--- CLIP Vision ---" && ls -lh models/clip_vision/

## 6) Checkpoint indir — IZOLE, curl ile

`curl -L` + Bearer header + URL token + UA + resume.

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
TOKEN = os.environ.get('CIVITAI_TOKEN', '')
%cd $WORKSPACE

if not TOKEN:
    print("⚠️  CIVITAI_TOKEN boş — CONFIG hücresine token girip o hücreyi çalıştır, sonra buraya dön.")
else:
    !mkdir -p models/checkpoints/sdxl
    !rm -f models/checkpoints/sdxl/waiIllustrious.safetensors

    MODEL_VERSION_ID = 2883731
    URL = f"https://civitai.com/api/download/models/{MODEL_VERSION_ID}?token={TOKEN}"
    OUT = "models/checkpoints/sdxl/waiIllustrious.safetensors"
    os.environ['DL_URL'] = URL
    os.environ['DL_OUT'] = OUT

    !curl -L --fail -C - \
        -H "Authorization: Bearer $CIVITAI_TOKEN" \
        -H "User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36" \
        -o "$DL_OUT" \
        "$DL_URL"

    if os.path.exists(OUT):
        size_gb = os.path.getsize(OUT) / 1024 / 1024 / 1024
        if size_gb < 1:
            print(f"\n⚠️  Dosya çok küçük ({size_gb*1024:.1f} MB). Token doğru mu?")
        else:
            print(f"\n✓ İndirme başarılı: {size_gb:.2f} GB")
    else:
        print("\n✗ Dosya hiç oluşmadı. Curl çıktısına bak.")

## 7) Dosya boyutu kontrolü

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')

checks = [
    (f'{WORKSPACE}/models/checkpoints/sdxl/waiIllustrious.safetensors', 1_000_000_000, 'Checkpoint (~6.5 GB)'),
    (f'{WORKSPACE}/models/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors', 100_000_000, 'IPAdapter (~800 MB)'),
    (f'{WORKSPACE}/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors', 1_000_000_000, 'CLIP Vision (~2.5 GB)'),
]

all_ok = True
print("="*60)
for path, min_size, desc in checks:
    if not os.path.exists(path):
        print(f"❌ YOK: {desc}")
        all_ok = False
    else:
        size_gb = os.path.getsize(path) / 1024 / 1024 / 1024
        ok = os.path.getsize(path) >= min_size
        mark = "✓" if ok else "⚠️"
        print(f"{mark} {desc:35s} {size_gb:.2f} GB")
        if not ok:
            all_ok = False
print("="*60)
print("✓ Hepsi hazır." if all_ok else "✗ Eksik var, ilgili indirme hücresini tekrar çalıştır.")

## 8) ComfyUI'yi arka planda başlat

Önce CONFIG'te yüklediğin referans görseli `ComfyUI/input/` klasörüne kopyalar (artık o klasör var), sonra ComfyUI'yi başlatır.

In [ ]:
import subprocess, time, socket, os, shutil
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
REF_FILENAME     = os.environ.get('REF_FILENAME', '')
REF_STAGING_PATH = os.environ.get('REF_STAGING_PATH', '')

# === Referans görseli ComfyUI/input/'a kopyala ===
if REF_STAGING_PATH and os.path.exists(REF_STAGING_PATH):
    input_dir = f'{WORKSPACE}/input'
    os.makedirs(input_dir, exist_ok=True)
    dest = f'{input_dir}/{REF_FILENAME}'
    shutil.copy(REF_STAGING_PATH, dest)
    print(f"✓ Referans kopyalandı: {dest}")
else:
    print("⚠️  Referans görsel staging'de bulunamadı — CONFIG hücresini tekrar çalıştır.")

# === ComfyUI başlat ===
def is_port_open(port):
    try:
        with socket.create_connection(('127.0.0.1', port), timeout=1):
            return True
    except (OSError, socket.timeout):
        return False

if is_port_open(8188):
    print("✓ ComfyUI zaten çalışıyor (port 8188)")
else:
    log_file = open('/tmp/comfyui.log', 'w')
    process = subprocess.Popen(
        ['python', 'main.py', '--dont-print-server'],
        cwd=WORKSPACE,
        stdout=log_file, stderr=subprocess.STDOUT
    )
    print(f"ComfyUI başlatılıyor (PID {process.pid})... port bekleniyor")
    for i in range(120):
        time.sleep(1)
        if is_port_open(8188):
            print(f"✓ ComfyUI hazır ({i+1} sn)")
            break
        if i % 10 == 9:
            print(f"   ... {i+1} sn")
    else:
        print("⚠️  120 sn'de başlamadı. Log:")
        !tail -30 /tmp/comfyui.log

## 9) Loop'la üret

Her aksiyon için workflow'u doldurur, API'ye gönderir, sonucu `outputs/01.png, 02.png...` olarak kaydeder.

**Node ID'ler:** 6 → QUALITY, 7 → NEGATIVE, 18 → CHARACTER, 19 → ACTION, 22 → BACKGROUND, 3 → KSampler, 9 → SaveImage, 12 → LoadImage

In [ ]:
import json, requests, time, shutil, os, copy

WORKSPACE     = os.environ.get('WORKSPACE', '/content/ComfyUI')
WORKFLOW_PATH = os.environ.get('WORKFLOW_PATH', '/content/workflow.json')
REF_FILENAME  = os.environ.get('REF_FILENAME', '')
API     = 'http://127.0.0.1:8188'
OUTPUTS = '/content/outputs'
os.makedirs(OUTPUTS, exist_ok=True)

if not os.path.exists(WORKFLOW_PATH):
    raise FileNotFoundError(f"Workflow bulunamadı: {WORKFLOW_PATH}. CONFIG hücresinden yükle.")
if not REF_FILENAME:
    raise RuntimeError("Referans görsel yok. CONFIG hücresinden yükle.")

with open(WORKFLOW_PATH) as f:
    TEMPLATE = json.load(f)

print(f"✓ Workflow: {WORKFLOW_PATH}  ({len(TEMPLATE)} node)")
print(f"✓ Ref: {REF_FILENAME}")

NODE_QUALITY    = "6"
NODE_NEGATIVE   = "7"
NODE_CHARACTER  = "18"
NODE_ACTION     = "19"
NODE_BACKGROUND = "22"
NODE_KSAMPLER   = "3"
NODE_SAVE       = "9"
NODE_REF_IMAGE  = "12"

def fill_workflow(template, quality, character, action, background, negative, seed, prefix, ref_filename):
    wf = copy.deepcopy(template)
    wf[NODE_QUALITY]['inputs']['text']    = quality
    wf[NODE_NEGATIVE]['inputs']['text']   = negative
    wf[NODE_CHARACTER]['inputs']['text']  = character
    wf[NODE_ACTION]['inputs']['text']     = action
    wf[NODE_BACKGROUND]['inputs']['text'] = background
    wf[NODE_KSAMPLER]['inputs']['seed']   = seed
    wf[NODE_SAVE]['inputs']['filename_prefix'] = prefix
    wf[NODE_REF_IMAGE]['inputs']['image'] = ref_filename
    return wf

def queue_prompt(workflow):
    r = requests.post(f"{API}/prompt", json={"prompt": workflow}, timeout=10)
    r.raise_for_status()
    return r.json()['prompt_id']

def wait_for_completion(prompt_id, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(f"{API}/history/{prompt_id}", timeout=10)
        if r.ok:
            data = r.json()
            if prompt_id in data:
                return data[prompt_id]
        time.sleep(2)
    raise TimeoutError(f"Prompt {prompt_id} timeout")

def get_output_images(history_entry):
    files = []
    for node_id, node_output in history_entry.get('outputs', {}).items():
        for img in node_output.get('images', []):
            sub = img.get('subfolder', '')
            path = f"{WORKSPACE}/output/{sub + '/' if sub else ''}{img['filename']}"
            files.append(path)
    return files

results = []
total = len(ACTIONS)
print(f"\n🎬 {total} görsel üretilecek\n")

for i, action in enumerate(ACTIONS, start=1):
    prefix = f"{i:02d}_run"
    print(f"[{i}/{total}] Üretiliyor: {action[:60]}{'...' if len(action) > 60 else ''}")
    t0 = time.time()
    try:
        wf = fill_workflow(TEMPLATE, QUALITY, CHARACTER, action, BACKGROUND, NEGATIVE, SEED, prefix, REF_FILENAME)
        pid = queue_prompt(wf)
        hist = wait_for_completion(pid)
        imgs = get_output_images(hist)
        if imgs:
            dest = f"{OUTPUTS}/{i:02d}.png"
            shutil.copy(imgs[0], dest)
            results.append({'idx': i, 'action': action, 'path': dest, 'time': time.time()-t0})
            print(f"   ✓ {dest}  ({time.time()-t0:.1f} sn)\n")
        else:
            print(f"   ⚠️  Görsel oluşmadı\n")
    except Exception as e:
        print(f"   ❌ Hata: {e}\n")

print(f"\n✓ Tamamlandı: {len(results)}/{total} görsel")

## 10) Sonuçları göster

In [ ]:
from IPython.display import Image, display, Markdown

for r in results:
    display(Markdown(f"### {r['idx']:02d} — {r['action']}"))
    display(Image(r['path'], width=500))
    display(Markdown(f"*{r['time']:.1f} sn*  •  `{r['path']}`\n\n---"))

## 11) Zip'le indir

In [ ]:
import shutil, os
from google.colab import files

zip_path = '/content/results.zip'
shutil.make_archive('/content/results', 'zip', '/content/outputs')
print(f"✓ Zip hazır: {zip_path}  ({os.path.getsize(zip_path)/1024**2:.1f} MB)")
files.download(zip_path)